# M16 · 05 — General post-training data (A0, prepared ahead of A3)

Decision: `docs/decisions/0025-own-llm-m16.md` §9 steps 1 (general SFT) and 3 (DPO). Produced by
`python -m llm.general_sft fetch` then `build` (raw parquet deleted after conversion, TRAINING.md §5b).

1. The datasets chosen, their licences and the ones rejected.
2. What survived conversion (format, language) per source.
3. Spot checks: language-ID quality and one example per source.
4. Token statistics with **our** tokenizer, the share over 4,096 tokens.
5. The mix: about **70 % Spanish by target tokens** (§9), written to `data/sft/general/mix.json` for the A3 SFT build.

### Selection and licences (checked on the Hub, 2026-11-15)

| Source (our name) | Hub repo | Licence | Language | Role |
|---|---|---|---|---|
| `smoltalk2-ml` | `HuggingFaceTB/smoltalk2`, SFT `smoltalk_multilingual_8languages_lang_5_no_think` | Apache-2.0 | es (filtered from 8 languages by LID) | SFT: SmolTalk prompts translated by Qwen and answered natively in Spanish |
| `aya` | `CohereLabs/aya_dataset` | Apache-2.0 | es + en (dataset label) | SFT: human-written |
| `openhermes-es` | `Iker/OpenHermes-2.5-Spanish` | Apache-2.0; a machine translation (TowerInstruct-13B) of `teknium/OpenHermes-2.5`, whose answers are GPT-4 outputs | es | SFT: Spanish volume |
| `smol-smoltalk` | `HuggingFaceTB/smol-smoltalk` | Apache-2.0 | en | SFT: SmolTalk subset made for models < 1B |
| `ultrafeedback` | `HuggingFaceH4/ultrafeedback_binarized` (train/test_prefs) | MIT | en | DPO (optional, §9.3) |

**Rejected:**
- `allenai/tulu-3-sft-mixture` (ODC-By, 1.4 GB): built for 8B+ models, heavy on maths and code,
  and its multilingual part is Aya, which we take directly. `smol-smoltalk` covers English better at our size.
- The original `HuggingFaceTB/smoltalk` (1M rows): `smol-smoltalk` is its curated subset for small models.
- The smoltalk2 `think` subsets: our model has no thinking mode (§4).

**Note on provenance.** `openhermes-es` and part of smoltalk carry outputs of proprietary models (GPT-4).
That is normal for open SFT data and allowed by these licences; it matters only if the model were
distributed commercially, which v1 doesn't do. Recorded here so the question is never re-derived.


In [1]:
import json, random, statistics
from collections import Counter
from llm import HOME
from llm.general_sft import OUT, SOURCES
from llm.sft_data import read_jsonl

for name, s in SOURCES.items():
    st = json.loads((OUT / f'{name}.stats.json').read_text())
    print(f"{name:14s} {s['kind']:3s} kept {st['kept']:>9,} of {st['rows']:>9,}  {st}")

smoltalk2-ml   sft kept    51,399 of   254,047  {'rows': 254047, 'kept': 51399, 'bad_format': 0, 'other_lang': 202648}
aya            sft kept     7,795 of   202,362  {'rows': 202362, 'kept': 7795, 'bad_format_or_other_lang': 194567, 'note': 'built before the bad_format/other_lang split was fixed; aya labels its language, so nearly all of these are other languages'}
openhermes-es  sft kept   985,802 of 1,000,000  {'rows': 1000000, 'kept': 985802, 'bad_format': 0, 'other_lang': 14198}
smol-smoltalk  sft kept   458,674 of   460,341  {'rows': 460341, 'kept': 458674, 'bad_format': 258, 'other_lang': 1409}
ultrafeedback  dpo kept    62,688 of    63,135  {'rows': 63135, 'kept': 62688, 'bad_format': 447, 'other_lang': 0}


### Language-ID spot check

`smoltalk2-ml` mixes 8 languages and keeps only what fastText lid.176 (the app's LID) calls Spanish.
Portuguese and Italian are the likely confusions, so: the LID confidence of a random sample of kept
examples, and how many of them look Portuguese by a crude marker count.

In [2]:
import fasttext
from llm.general_sft import lid_path
m = fasttext.load_model(str(lid_path()))
rng = random.Random(0)
rows = list(read_jsonl(OUT / 'smoltalk2-ml.jsonl'))
sample = rng.sample(rows, 2000)
conf, pt = [], 0
for ex in sample:
    t = ' '.join(x['content'] for x in ex['messages'] if x['role'] != 'system')[:1000].replace('\n', ' ')
    lab, p = m.predict(t, k=2)
    conf.append(p[0])
    w = t.lower().split()
    pt += sum(x in ('não', 'você', 'são', 'também', 'então') for x in w) > sum(x in ('no', 'usted', 'son', 'también', 'entonces') for x in w)
print(f'kept smoltalk2-ml examples: {len(rows):,}')
print(f'LID p(es): median {statistics.median(conf):.3f}, share < 0.9: {sum(c < 0.9 for c in conf) / len(conf):.1%}')
print(f'look Portuguese by markers: {pt} / {len(sample)}')

kept smoltalk2-ml examples: 51,399
LID p(es): median 0.969, share < 0.9: 5.9%
look Portuguese by markers: 0 / 2000


In [3]:
# One example per source, truncated, to eyeball quality.
for name, s in SOURCES.items():
    ex = next(read_jsonl(OUT / f'{name}.jsonl'))
    user = [x for x in ex['messages'] if x['role'] == 'user'][-1]['content']
    ans = ex['messages'][-1]['content'] if s['kind'] == 'sft' else ex['chosen']
    print(f"=== {name} ({ex['lang']}) ===\nUSER: {user[:300]}\nASSISTANT: {ans[:400]}\n")

=== smoltalk2-ml (es) ===
USER: ¿Cuáles son algunos síntomas comunes de un resfriado y cuánto tiempo suelen durar? Tu respuesta debe contener al menos 3 puntos con viñetas. Usa los puntos con viñetas de markdown, por ejemplo: * Este es el punto 1.
ASSISTANT: Los síntomas comunes de un resfriado suelen incluir:

* Congestión nasal o secreción nasal
* Estornudos y tos
* Dolor de garganta y malestar general

La duración de los síntomas de un resfriado puede variar, pero generalmente suelen durar:

* Entre 7 y 10 días, aunque algunos síntomas pueden persistir hasta 2 semanas.

=== aya (es) ===
USER: Sentencia 1: Este fue el décimo intento de abastecer el oasis por tierra, después de la derrota del ejército italiano en la Operación Compas y la retirada de Cirenaica. Sentencia 2: Este fue el último intento italiano de proporcionar el oasis después de la derrota del ejército en la Operación Compas
ASSISTANT: Si, el décimo es el último intento de abastecer el oasis por tierra del ejercito ital

### Token statistics (our tokenizer)

A random sample of up to 5,000 examples per source, encoded exactly as `llm.sft_data` encodes them
(ChatML, assistant-only mask). *Target tokens* are the masked ones, the tokens the loss sees.

In [4]:
from llm.prepare_data import tokenizer_path
from llm.sft_data import encode_example
from llm.tokenizer import load_tokenizer
tok = load_tokenizer(tokenizer_path())

def sft_view(ex, kind):
    return ex['messages'] if kind == 'sft' else ex['messages'] + [{'role': 'assistant', 'content': ex['chosen']}]

tstats = {}
for name, s in SOURCES.items():
    rows = list(read_jsonl(OUT / f'{name}.jsonl'))
    by_lang = Counter(ex['lang'] for ex in rows)
    sample = random.Random(0).sample(rows, min(5000, len(rows)))
    lens, tg = [], []
    for ex in sample:
        ids, mask = encode_example(tok, sft_view(ex, s['kind']))
        lens.append(len(ids)); tg.append(sum(mask))
    over = sum(l > 4096 for l in lens) / len(lens)
    tstats[name] = dict(n=len(rows), by_lang=dict(by_lang), mean_len=statistics.mean(lens),
                        mean_target=statistics.mean(tg), over_4096=over,
                        total_target_M=statistics.mean(tg) * len(rows) * (1 - over) / 1e6)
    print(f"{name:14s} n={len(rows):>9,} {dict(by_lang)}  mean {tstats[name]['mean_len']:6.0f} tok, "
          f"target {tstats[name]['mean_target']:6.0f}, >4096 {over:.2%}, ~{tstats[name]['total_target_M']:,.0f}M target tok")

smoltalk2-ml   n=   51,399 {'es': 51399}  mean    510 tok, target    412, >4096 0.00%, ~21M target tok


aya            n=    7,795 {'es': 3854, 'en': 3941}  mean    141 tok, target     92, >4096 0.00%, ~1M target tok


openhermes-es  n=  985,802 {'es': 985802}  mean    455 tok, target    268, >4096 0.04%, ~264M target tok


smol-smoltalk  n=  458,674 {'en': 458674}  mean    969 tok, target    697, >4096 0.82%, ~317M target tok


ultrafeedback  n=   62,688 {'en': 62688}  mean    542 tok, target    337, >4096 0.04%, ~21M target tok


### The mix

Rules (§9.1): about 70 % Spanish **by target tokens**, sequences ≤ 4,096 tokens (dropped by `llm.sft_data build`).

- All the native-Spanish and human data is used whole (weight 1): `smoltalk2-ml`, `aya`.
- `openhermes-es` (translated) fills the rest of the Spanish budget.
- `smol-smoltalk` is subsampled so English is 30 %.
- **Budget:** ~150M target tokens in total. For a 110–330M model that is several epochs' worth of
  instruction signal compared with published small-model SFT (SmolLM2-135M/360M used smol-smoltalk,
  ~0.5M examples), while the task SFT (notebook 03) stays the priority. A3/B2 can rescale it.
- `aya` holds both languages in one file; its English part is counted on the English side.
- `ultrafeedback` is DPO only, not in the SFT mix.

In [5]:
BUDGET_M, ES_SHARE = 150, 0.70
t = tstats
def lang_part(name, lang):
    return t[name]['total_target_M'] * t[name]['by_lang'].get(lang, 0) / t[name]['n']

es_fixed = t['smoltalk2-ml']['total_target_M'] + lang_part('aya', 'es')
en_fixed = lang_part('aya', 'en')
es_left = max(0.0, BUDGET_M * ES_SHARE - es_fixed)
en_left = max(0.0, BUDGET_M * (1 - ES_SHARE) - en_fixed)
w = {'smoltalk2-ml': 1.0, 'aya': 1.0,
     'openhermes-es': round(min(1.0, es_left / t['openhermes-es']['total_target_M']), 4),
     'smol-smoltalk': round(min(1.0, en_left / t['smol-smoltalk']['total_target_M']), 4)}
es = es_fixed + w['openhermes-es'] * t['openhermes-es']['total_target_M']
en = en_fixed + w['smol-smoltalk'] * t['smol-smoltalk']['total_target_M']
print('weights:', w)
print(f'target tokens: es {es:,.0f}M + en {en:,.0f}M = {es + en:,.0f}M  ->  es share {es / (es + en):.1%}')
mix = {'budget_target_tokens_M': BUDGET_M, 'es_share': round(es / (es + en), 4), 'weights': w,
       'files': {n: str(OUT / f'{n}.jsonl') for n in w},
       'sft_data_args': [f"{OUT / (n + '.jsonl')}:{w[n]}" for n in w],
       'dpo_files': [str(OUT / 'ultrafeedback.jsonl')],
       'token_stats': tstats}
(OUT / 'mix.json').write_text(json.dumps(mix, indent=1))
print('written', OUT / 'mix.json')
print('A3 build: python -m llm.sft_data build --out <dir> --seq-len 4096 ' + ' '.join(mix['sft_data_args']))

weights: {'smoltalk2-ml': 1.0, 'aya': 1.0, 'openhermes-es': 0.3159, 'smol-smoltalk': 0.1408}
target tokens: es 105M + en 45M = 150M  ->  es share 70.0%
written /home/pablo/yts-experiments/m16-llm/data/sft/general/mix.json
A3 build: python -m llm.sft_data build --out <dir> --seq-len 4096 /home/pablo/yts-experiments/m16-llm/data/sft/general/smoltalk2-ml.jsonl:1.0 /home/pablo/yts-experiments/m16-llm/data/sft/general/aya.jsonl:1.0 /home/pablo/yts-experiments/m16-llm/data/sft/general/openhermes-es.jsonl:0.3159 /home/pablo/yts-experiments/m16-llm/data/sft/general/smol-smoltalk.jsonl:0.1408


### DPO pairs (ultrafeedback)

How many pairs, the score margin, and the train/val split carried from the Hub (`test_prefs` → val).

In [6]:
pairs = list(read_jsonl(OUT / 'ultrafeedback.jsonl'))
marg = [p['score_chosen'] - p['score_rejected'] for p in pairs if p.get('score_chosen') is not None]
print(Counter(p['split'] for p in pairs), f'median score margin {statistics.median(marg):.2f}')

Counter({'train': 60700, 'val': 1988}) median score margin 1.00


### Conclusions (run of 2026-11-15)

- **Kept after conversion:** smoltalk2-ml 51,399 (es only; 20 % of the 8-language set), aya 7,795
  (3,854 es + 3,941 en of 202k rows in 70+ languages), openhermes-es 985,802, smol-smoltalk 458,674,
  ultrafeedback 62,688 pairs (60,700 train / 1,988 val).
- **LID is reliable here:** on 2,000 kept smoltalk2-ml examples the median p(es) is 0.969, 5.9 % fall
  below 0.9, and none look Portuguese by marker words.
- **Length:** almost nothing exceeds 4,096 tokens (0–0.8 %), so the §9 filter costs nothing.
- **Spanish native data is small** (~22M target tokens: smoltalk2-ml + aya). The translated
  openhermes-es (~264M) is what makes 70 % Spanish possible; its translations read naturally in the
  samples, but it is the first thing to down-weight if A3 shows translationese.
- **Mix (mix.json):** smoltalk2-ml 1.0, aya 1.0, openhermes-es 0.3159, smol-smoltalk 0.1408 →
  105M es + 45M en = **150M target tokens, 70.0 % Spanish**. The A3 build command is printed above.
- **DPO:** ultrafeedback is English only and has a median score margin of 1.0; per §9.3 it is mixed in
  only if it helps on val.
- **Next (A3):** build the general SFT dir with that command, then task SFT (notebook 03 data) with
  10–20 % replay from this mix.
